# ***Install Dependencies & Import Libraries***

In [ ]:
!pip install transformers datasets seqeval

import numpy as np
import tensorflow as tf
from datasets import load_dataset
from transformers import BertTokenizerFast, TFBertModel
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Bidirectional, LSTM, TimeDistributed, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from seqeval.metrics import classification_report
from IPython.display import display, HTML

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 19.4 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=7e22a615b58389d262492fcf99d8bacf53bde13e75476f9a41781f73f2401a62
  Stored in directory: /root/.cache/pip/wheels/bc/92/f0/243288f899c2eacdfa8c5f9aede4c71a9bad0ee26a01dc5ead
Successfully built seqeval
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency res

# ***Load CoNLL-2003 Dataset***

In [ ]:
# Load the CoNLL-2003 dataset
dataset = load_dataset("conll2003")
train_data = dataset["train"]
val_data = dataset["validation"]
test_data = dataset["test"]

# NER tag names
tag_names = dataset["train"].features["ner_tags"].feature.names
n_tags = len(tag_names)
tag2idx = {tag: idx for idx, tag in enumerate(tag_names)}
idx2tag = {idx: tag for tag, idx in tag2idx.items()}

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/12.3k [00:00<?, ?B/s]

conll2003.py:   0%|          | 0.00/9.57k [00:00<?, ?B/s]

The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

# ***Preprocess Data Using BERT Tokenizer***

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")
max_len = 128

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding='max_length',
        max_length=max_len,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx] if tag_names[label[word_idx]].startswith("I-") else -100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Tokenize datasets
tokenized_train = train_data.map(tokenize_and_align_labels, batched=True)
tokenized_val = val_data.map(tokenize_and_align_labels, batched=True)
tokenized_test = test_data.map(tokenize_and_align_labels, batched=True)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

# ***Prepare TensorFlow Datasets***

In [ ]:
def to_tf_dataset(tokenized_dataset):
    labels = np.array(tokenized_dataset["labels"])
    labels[labels == -100] = 0
    return tf.data.Dataset.from_tensor_slices((
        {
            "input_ids": np.array(tokenized_dataset["input_ids"]),
            "attention_mask": np.array(tokenized_dataset["attention_mask"])
        },
        to_categorical(labels, num_classes=n_tags) # Use modified labels
    )).batch(16)

train_ds = to_tf_dataset(tokenized_train)
val_ds = to_tf_dataset(tokenized_val)
test_ds = to_tf_dataset(tokenized_test)

print("Train dataset shape:", train_ds.element_spec)
print("Validation dataset shape:", val_ds.element_spec)
print("Test dataset shape:", test_ds.element_spec)

Train dataset shape: ({'input_ids': TensorSpec(shape=(None, 128), dtype=tf.int64, name=None), 'attention_mask': TensorSpec(shape=(None, 128), dtype=tf.int64, name=None)}, TensorSpec(shape=(None, 128, 9), dtype=tf.float64, name=None))
Validation dataset shape: ({'input_ids': TensorSpec(shape=(None, 128), dtype=tf.int64, name=None), 'attention_mask': TensorSpec(shape=(None, 128), dtype=tf.int64, name=None)}, TensorSpec(shape=(None, 128, 9), dtype=tf.float64, name=None))
Test dataset shape: ({'input_ids': TensorSpec(shape=(None, 128), dtype=tf.int64, name=None), 'attention_mask': TensorSpec(shape=(None, 128), dtype=tf.int64, name=None)}, TensorSpec(shape=(None, 128, 9), dtype=tf.float64, name=None))


# ***Define BERT + BiLSTM Model***

In [ ]:
# Define BERT + BiLSTM Model
input_ids = Input(shape=(max_len,), dtype=tf.int32, name='input_ids')
attention_mask = Input(shape=(max_len,), dtype=tf.int32, name='attention_mask')

# Load BERT
bert_model = TFBertModel.from_pretrained("bert-base-cased")

# Wrap the BERT call in a Lambda layer to convert KerasTensors to TensorFlow Tensors
# Specify the output shape for the Lambda layer
bert_output = tf.keras.layers.Lambda(
    lambda x: bert_model(input_ids=x[0], attention_mask=x[1]).last_hidden_state,
    output_shape=(max_len, bert_model.config.hidden_size)  # Add output_shape
)([input_ids, attention_mask])

# Add BiLSTM and TimeDistributed layer
lstm = Bidirectional(LSTM(64, return_sequences=True))(bert_output)
output = TimeDistributed(Dense(n_tags, activation="softmax"))(lstm)

# Build and compile model
model = Model(inputs=[input_ids, attention_mask], outputs=output)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_ids           │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_mask      │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 128, 768)  │          0 │ input_ids[0][0],  │
│                     │                   │            │ attention_mask[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 128, 128)  │    426,496 │ lambda[0][0]      │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 128, 9)    │      1,161 │ bidirectional[0]… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 427,657 (1.63 MB)

 Trainable params: 427,657 (1.63 MB)

 Non-trainable params: 0 (0.00 B)

# ***Train the Model***

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
    ModelCheckpoint("bert_bilstm_ner.h5", save_best_only=True)
]

history = model.fit(train_ds, validation_data=val_ds, epochs=3, callbacks=callbacks)

Epoch 1/3
878/878 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step - accuracy: 0.9680 - loss: 0.1314

878/878 ━━━━━━━━━━━━━━━━━━━━ 206s 209ms/step - accuracy: 0.9680 - loss: 0.1313 - val_accuracy: 0.9943 - val_loss: 0.0226
Epoch 2/3
878/878 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step - accuracy: 0.9951 - loss: 0.0178

878/878 ━━━━━━━━━━━━━━━━━━━━ 176s 201ms/step - accuracy: 0.9951 - loss: 0.0178 - val_accuracy: 0.9957 - val_loss: 0.0159
Epoch 3/3
878/878 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step - accuracy: 0.9968 - loss: 0.0120

878/878 ━━━━━━━━━━━━━━━━━━━━ 186s 212ms/step - accuracy: 0.9968 - loss: 0.0120 - val_accuracy: 0.9961 - val_loss: 0.0138


# ***Evaluate the Model***

In [ ]:
# Predict
preds = model.predict(test_ds)
y_pred = np.argmax(preds, axis=-1)
y_true = np.argmax(np.concatenate([y for x, y in test_ds], axis=0), axis=-1)

# Convert to tag names
true_tags, pred_tags = [], []
for i in range(len(y_true)):
    t_seq, p_seq = [], []
    for j in range(len(y_true[i])):
        if y_true[i][j] != 0:  # skip padding
            t_seq.append(idx2tag[y_true[i][j]])
            p_seq.append(idx2tag[y_pred[i][j]])
    true_tags.append(t_seq)
    pred_tags.append(p_seq)

# Report
print(classification_report(true_tags, pred_tags))


216/216 ━━━━━━━━━━━━━━━━━━━━ 40s 174ms/step
              precision    recall  f1-score   support

         LOC       0.87      0.88      0.88      1666
        MISC       0.74      0.76      0.75       702
         ORG       0.82      0.84      0.83      1661
         PER       0.96      0.93      0.95      1615

   micro avg       0.87      0.87      0.87      5644
   macro avg       0.85      0.85      0.85      5644
weighted avg       0.87      0.87      0.87      5644



# ***Visualize Prediction on a Sentence***

In [ ]:
def color_ner_output(tokens, tags):
    colors = {
        'O': 'black',
        'B-PER': 'red', 'I-PER': 'salmon',
        'B-LOC': 'green', 'I-LOC': 'lightgreen',
        'B-ORG': 'blue', 'I-ORG': 'lightblue',
        'B-MISC': 'purple', 'I-MISC': 'violet'
    }
    html = ""
    for token, tag in zip(tokens, tags):
        color = colors.get(tag, 'gray')
        html += f"<span style='color:{color}; font-weight:bold'>{token}</span> "
    return html

def predict_and_visualize(sentence):
    tokens = sentence.split()
    inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="tf", padding='max_length', truncation=True, max_length=max_len)
    # Select only 'input_ids' and 'attention_mask'
    preds = model.predict({"input_ids": inputs["input_ids"], "attention_mask": inputs["attention_mask"]})
    pred_ids = np.argmax(preds, axis=-1)[0][:len(tokens)]
    tags = [idx2tag[i] for i in pred_ids]

    print("Word\tPredicted Tag")
    print("-"*25)
    for w, t in zip(tokens, tags):
        print(f"{w:10}\t{t}")
    display(HTML(color_ner_output(tokens, tags)))

# Example
predict_and_visualize("Barack Obama visited Paris in 2009 .")

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Word	Predicted Tag
-------------------------
Barack    	O
Obama     	B-PER
visited   	I-PER
Paris     	O
in        	B-LOC
2009      	O
.         	O


# ***Build a Smart Resume Parser App Using Gradio***

# ***Install Gradio***

In [ ]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.9/322.9 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 6.3 MB/s eta 0:00:00


# ***Define Prediction + Visualization Functions***

In [ ]:
def predict_entities(text):
    tokens = text.split()
    inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="tf", padding='max_length', truncation=True, max_length=max_len)

    # Convert BatchEncoding to a dict of tensors
    input_dict = {
        "input_ids": inputs["input_ids"],
        "token_type_ids": inputs["token_type_ids"],
        "attention_mask": inputs["attention_mask"]
    }

    preds = model.predict(input_dict)  # ✅ pass as a dict
    pred_ids = np.argmax(preds, axis=-1)[0][:len(tokens)]
    tags = [idx2tag[i] for i in pred_ids]

    html = ""
    colors = {
        'O': 'black',
        'B-PER': 'red', 'I-PER': 'salmon',
        'B-LOC': 'green', 'I-LOC': 'lightgreen',
        'B-ORG': 'blue', 'I-ORG': 'lightblue',
        'B-MISC': 'purple', 'I-MISC': 'violet'
    }
    for token, tag in zip(tokens, tags):
        color = colors.get(tag, 'gray')
        html += f"<span style='color:{color}; font-weight:bold'>{token} ({tag})</span> "

    return html


# ***Create and Launch Gradio App***

In [ ]:
iface = gr.Interface(
    fn=predict_entities,
    inputs=gr.Textbox(lines=4, placeholder="Paste your resume text or sentence here..."),
    outputs=gr.HTML(),
    title="Smart Resume Entity Extractor",
    description="Enter resume text. This model will extract entities like names, locations, organizations, and more using BERT + BiLSTM."
)

iface.launch(debug= True)

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://743151509648b3bff3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7863 <> https://743151509648b3bff3.gradio.live
